In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

import warnings

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

from spatial_tcr.clustering import (
    leiden_unique,
)
from spatial_tcr.tcr import get_tcr_genes

# Suppress performance warnings

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
def flatten_dict(d):
    return [i for items in d.values() for i in items]

In [ ]:
adata = sc.read_h5ad("data/xenium/processed/03-kidney_tcr_classified.h5ad")
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()
adata.X = adata.layers["counts"].copy()
adata, adata.X.max()

In [ ]:
mask = (
    adata[
        :, ["CD3D", "CD3E", "CD3G", "CD8A", "TRBC1", "TRBC2", "TRDC", "TRGC1", "TRGC2"]
    ]
    .X.toarray()
    .sum(axis=1)
    > 0
)
adata[mask].obs["cell_type_no_tcr"].value_counts()

In [ ]:
adata.obs["slide"].value_counts()

In [ ]:
ad_t = adata[adata.obs["cell_type_no_tcr"] == "T"].copy()
ad_t

In [ ]:
av_genes, bv_genes, dv_genes, gv_genes, tv_genes = get_tcr_genes(ad_t)

In [ ]:
markers = {
    "MAIT": ["TRAV1-2"],
    "gdT": ["TRDC", "TRGC1", "TRGC2"],
    "CD8": [
        "CD8A",
        "GZMK",
        "GZMH",
        "GZMB",
    ],
    "CD4": [
        "CD4",
        "IL23A",
    ],
}
markers_flattend = [gene for genes in markers.values() for gene in genes]
markers_flattend = []

In [ ]:
var_names = [v for v in ad_t.var_names if v not in tv_genes]
var_names = [v for v in var_names if v not in markers_flattend] + markers_flattend

ad_t = ad_t[:, var_names].copy()
print(ad_t.shape)
# remove cells with zero expression
sc.pp.filter_cells(ad_t, min_genes=1)
print(ad_t.shape)

In [ ]:
sc.pp.normalize_total(ad_t)
sc.pp.log1p(ad_t)

In [ ]:
sc.pp.highly_variable_genes(ad_t, n_top_genes=100, batch_key="sample")

In [ ]:
sc.pl.highly_variable_genes(ad_t)

In [ ]:
reduced_var_names = ad_t.var_names[ad_t.var["highly_variable"]].tolist()
reduced_var_names = [
    v for v in reduced_var_names if v not in markers_flattend
] + markers_flattend
ad_t = ad_t[:, reduced_var_names].copy()
ad_t

In [ ]:
# remove cells with zero expression
print(ad_t.shape)
sc.pp.filter_cells(ad_t, min_genes=5)
print(ad_t.shape)

In [ ]:
ad_t.X = ad_t.layers["counts"].copy()
sc.pp.normalize_total(ad_t)
sc.pp.log1p(ad_t)
sc.tl.pca(ad_t, n_comps=50)

In [ ]:
# sc.pl.pca_variance_ratio(ad_t, n_pcs=50, log=True)
# sc.pl.pca(ad_t, color=["slide"])

In [ ]:
ad_t.obsm["X_pca"][0]

In [ ]:
ad_t.obsm["X_pca"][0]

In [ ]:
leiden_unique(
    ad_t, use_rep="X_pca", resolution=2.0, n_neighbors=50, key_added="leiden_tcell"
)

In [ ]:
sc.pp.neighbors(ad_t, use_rep="X_pca", n_neighbors=15)
sc.tl.umap(ad_t)
sc.pl.umap(
    ad_t,
    color="leiden_tcell",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

## Add back all original genes

In [ ]:
ad_t_full = adata[ad_t.obs_names].copy()
sc.pp.normalize_total(ad_t_full)
sc.pp.log1p(ad_t_full)
ad_t_full.obsm["X_pca"] = ad_t.obsm["X_pca"]
ad_t_full.obs["leiden_tcell"] = ad_t.obs["leiden_tcell"].copy()
sc.pp.neighbors(ad_t_full, use_rep="X_pca", n_neighbors=15)
sc.tl.umap(ad_t_full)

In [ ]:
ad_t_full.obs["leiden_tcell"].value_counts()

In [ ]:
markers = {
    "T": ["CD3D", "CD3E", "CD3G"],
    "MAIT": ["TRAV1-2", "KLRB1", "CCR6"],
    "gdT": ["TRDC", "TRGC1", "TRGC2"],
    "CD8": [
        "CD8A",
        "GZMK",
        "GZMH",
        "GZMB",
    ],
    "CD4": ["CD4", "TRBC2"],
    "TFH": ["CXCL13", "CXCR5", "BCL6"],
    "NKT": ["GNLY", "NKG7"],
    "TH1": ["IFNG", "TNF"],
    "TH17": ["IL17A", "RORC", "IL23R"],
    "TCM": ["CCR7", "SELL"],
    "TRM": ["CD69", "ITGAE", "CXCR6"],
    "Treg": ["FOXP3", "IKZF2"],
    "NK": ["KLRC1"],
    "Myeloid": ["CD74", "LYZ", "HLA-DRA"],
    "Prolif.": ["STMN1"],
    "Fib": ["COL1A1", "ACTA2"],
}
sc.pl.dotplot(ad_t_full, markers, groupby="leiden_tcell", standard_scale="var")

In [ ]:
# sc.pl.dotplot(adata, markers, groupby="cell_type_no_tcr", standard_scale="var")

In [ ]:
# check differential gene expression
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(ad_t_full, groupby="leiden_tcell", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(
    ad_t_full, groupby="leiden_tcell", standard_scale="var", n_genes=5
)

In [ ]:
# extract all these markers
degs = {
    g: sc.get.rank_genes_groups_df(ad_t_full, group=g).head(8).names.tolist()
    for g in ad_t_full.obs["leiden_tcell"].value_counts().index
}
degs

In [ ]:
# # extract all these markers
# degs = {
#     g: sc.get.rank_genes_groups_df(ad_t_full, group=g).tail(5).names.tolist()
#     for g in ad_t_full.obs["leiden_tcell"].unique()
# }
# degs

In [ ]:
annots_res_20 = dict.fromkeys(ad_t_full.obs["leiden_tcell"].value_counts().index, "")
annots_res_20 = {
    "21": "FIB",
    "1": "CD8+",
    "14": "CD4+",  # Th17 like
    "6": "CD8+",
    "15": "EC",  # mixed
    "7": "CD8+",  # mixed
    "17": "FIB",
    "20": "CD4+",  # maybe Mac
    "19": "Mac",
    "0": "NK/NKT",
    "12": "EC",
    "18": "CD4+",
    "23": "Tregs",
    "4": "CD4+",
    "13": "CD4+",
    "8": "MAIT",
    "10": "FIB",
    "3": "CD4+",
    "9": "EC",
    "11": "gdT",
    "16": "CD4+",  # mixed
    "5": "CD4+",
    "24": "TFH",
    "2": "CD4+",
    "22": "CD8+",  # mixed with B cells
}

ad_t_full.obs["tcell_subtype"] = ad_t_full.obs["leiden_tcell"].map(annots_res_20)
sc.pl.dotplot(
    ad_t_full, markers, groupby="tcell_subtype", standard_scale="var", dendrogram=False
)

In [ ]:
sc.tl.rank_genes_groups(ad_t_full, groupby="tcell_subtype", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(
    ad_t_full, groupby="tcell_subtype", standard_scale="var", n_genes=5, dendrogram=True
)

In [ ]:
plt.rcParams["figure.figsize"] = (4, 4)
ax = sc.pl.umap(
    ad_t_full,
    color="tcell_subtype",
    # Setting a smaller point size to get prevent overlap
    size=4,
    title="T cell subtypes",
    show=False,
)
# despine
sns.despine(ax=ax)

In [ ]:
plt.rcParams["figure.figsize"] = (2.3, 2.3)
colors = ["CD8A", "CD4", "TRDC", "GNLY", "FOXP3", "CXCL13"]
axs = sc.pl.umap(
    ad_t_full,
    color=colors,
    # Setting a smaller point size to get prevent overlap
    size=5,
    cmap="Blues",
    vmax=1.5,
    ncols=3,
    hspace=0.3,
    show=False,
)
for ax in axs:
    # despine
    sns.despine(ax=ax)

In [ ]:
# ad_t_full.write_h5ad("data/xenium/processed/04-kidney_tcr_tsub_only.h5ad")

In [ ]:
# inject fixed annotations to ensure reproducibility
import pandas as pd

tcell_subtype_df = pd.read_csv("data/xenium/tcell_subtype.csv", index_col=0)
adata.obs.loc[tcell_subtype_df.index, "tcell_subtype"] = tcell_subtype_df[
    "tcell_subtype"
]

# # save into anndata
# adata.obs["tcell_subtype"] = adata.obs["cell_type_no_tcr"].astype(str)
# adata.obs.loc[ad_t_full.obs_names, "tcell_subtype"] = ad_t_full.obs["tcell_subtype"]
# # replace T with unknown
# adata.obs.loc[adata.obs["tcell_subtype"] == "T", "tcell_subtype"] = "unknown"

adata.obs["tcell_subtype"].value_counts()

In [ ]:
sc.pl.dotplot(adata, markers, groupby="tcell_subtype", standard_scale="var")

In [ ]:
adata.write_h5ad("data/xenium/processed/04-kidney_tcr_tsub.h5ad")